In [2]:
import os
import pandas as pd
import numpy as np
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build
from dotenv import load_dotenv
from pycoingecko import CoinGeckoAPI
import warnings

warnings.filterwarnings("ignore")

# Load environment variables
load_dotenv()

# Google Sheets configuration
SCOPES = ['https://www.googleapis.com/auth/spreadsheets.readonly']
SERVICE_ACCOUNT_FILE = os.getenv('SERVICE_ACCOUNT_FILE')
SPREADSHEET_ID = '1156S3pm9vQ2JP5bE0yvpj_Y8FE55ecovHS_JvIys5dc'

# Authenticate and build the service
creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=SCOPES)
service = build('sheets', 'v4', credentials=creds)

def get_allocation_data(sheet_name):
    """
    Pulls allocation data from the specified Google Sheets sheet.
    
    Parameters:
        sheet_name (str): The name of the sheet (e.g. "EXC", "HB", or "LC")
    
    Returns:
        pd.DataFrame: DataFrame built from the sheet values (header in first row)
    """
    range_name = f'{sheet_name}!A:AAZ'
    result = service.spreadsheets().values().get(
        spreadsheetId=SPREADSHEET_ID,
        range=range_name
    ).execute()
    values = result.get('values', [])
    if not values:
        print(f'No data found in sheet {sheet_name}.')
        return None
    # Assume the first row is the header (tickers and possibly a "date" column)
    df = pd.DataFrame(values[1:], columns=values[0])
    return df

def parse_allocation_row(df):
    """
    Converts the latest (last) row of the allocation DataFrame to a dictionary of ticker: allocation,
    and extracts the start date.
    """
    allocation_date = None
    allocations = df.iloc[-1]
    allocation_dict = {}
    
    # Look for date column with case-insensitive match (including Portuguese "Data")
    date_column = next((col for col in df.columns if col.lower() in ['date', 'data']), None)
    if date_column:
        try:
            allocation_date = pd.to_datetime(allocations[date_column])
            print(f"Found date: {allocation_date}")
        except Exception as e:
            print(f"Error parsing date from column {date_column}:", e)
    else:
        print("No date column found in the sheet (looked for 'date' or 'data')")
    
    # Process allocations
    for ticker, value in allocations.items():
        if ticker.lower() not in ['date', 'data']:  # Skip date column
            if isinstance(value, str):
                value = value.strip()
                if value.endswith('%'):
                    try:
                        allocation = float(value.replace('%', '')) / 100
                    except ValueError:
                        allocation = 0.0
                else:
                    try:
                        allocation = float(value)
                    except ValueError:
                        allocation = 0.0
            else:
                try:
                    allocation = float(value)
                except Exception:
                    allocation = 0.0
            allocation_dict[ticker.upper()] = allocation
    
    return allocation_dict, allocation_date

def get_latest_allocations():
    """
    Pulls the latest allocation data for the three portfolios.
    The portfolios are defined by sheet names:
      carteira_EXC -> sheet "EXC"
      carteira_HB  -> sheet "HB"
      carteira_LC  -> sheet "LC"
    
    Returns:
         dict: Dictionary with keys 'carteira_EXC', 'carteira_HB', and 'carteira_LC'
               and each value is a tuple (allocation_dict, allocation_date).
    """
    portfolios = {
        'carteira_EXC': 'EXC',
        'carteira_HB': 'HB',
        'carteira_LC': 'LC'
    }
    all_allocations = {}
    for portfolio_key, sheet_name in portfolios.items():
        df = get_allocation_data(sheet_name)
        if df is not None:
            allocation_dict, allocation_date = parse_allocation_row(df)
            all_allocations[portfolio_key] = (allocation_dict, allocation_date)
    return all_allocations

def get_ticker_mapping(portfolio_assets):
    """
    Uses the CoinGecko API to retrieve the coins list and creates a mapping 
    from coin ticker (upper-case) to coin id.
    """
    cg = CoinGeckoAPI()
    coins_list = cg.get_coins_list()
    coins_df = pd.DataFrame(coins_list)
    
    # Define known mappings for ambiguous tickers
    known_mappings = {
        'VIRTUAL': 'virtual-protocol',
        'HYPE': 'hyperliquid',
        'YNE': 'yesnoerror'
    }
    
    # Create mapping by first checking known mappings, then filtering by portfolio_assets
    mapping = {}
    
    # Add known mappings first
    for ticker, coin_id in known_mappings.items():
        if coin_id in portfolio_assets:
            mapping[ticker] = coin_id
    
    # Add remaining mappings from CoinGecko
    filtered_coins_df = coins_df[coins_df['id'].isin(portfolio_assets)]
    for _, row in filtered_coins_df.iterrows():
        ticker = row['symbol'].upper()
        if ticker not in mapping:  # Don't override known mappings
            mapping[ticker] = row['id']
    
    return mapping

def build_dataframe(portfolio_assets, allocation_dict=None):
    """
    Builds a price close DataFrame using either all assets in portfolio_assets list
    or only the tickers with non-zero allocation if allocation_dict is provided.
    """
    mapping = get_ticker_mapping(portfolio_assets)
    price_data = {}
    
    if allocation_dict is not None:
        assets_to_process = {ticker: alloc for ticker, alloc in allocation_dict.items() if alloc > 0}
    else:
        reverse_mapping = {v: k for k, v in mapping.items()}
        assets_to_process = {reverse_mapping.get(asset_id, asset_id): 1 for asset_id in portfolio_assets}
    
    # Process each asset
    for ticker in assets_to_process:
        if ticker in mapping:
            coin_id = mapping[ticker]
            file_path = os.path.join("micro", "assetData", f"{coin_id}.csv")
            if os.path.exists(file_path):
                df = pd.read_csv(file_path)
                if 'date' in df.columns and 'close' in df.columns:
                    df['date'] = pd.to_datetime(df['date'])
                    df = df.drop_duplicates(subset=['date'])  # Remove duplicate dates
                    df.set_index('date', inplace=True)
                    price_data[ticker] = df['close']
                else:
                    print(f"Required columns not found in {file_path}")
            else:
                print(f"File not found: {file_path}")
        else:
            print(f"Ticker {ticker} not found in ticker mapping.")
    
    if not price_data:
        return pd.DataFrame()
    
    # Create DataFrame with all series and ensure index is unique
    price_df = pd.DataFrame(price_data)
    price_df = price_df[~price_df.index.duplicated(keep='first')]  # Keep first occurrence of duplicate indices
    
    # Sort index to ensure chronological order
    price_df.sort_index(inplace=True)
    
    # For each column (asset), fill NaN values with the first available price
    for column in price_df.columns:
        first_valid_price = price_df[column].first_valid_index()
        if first_valid_price is not None:
            price_df[column].fillna(price_df[column][first_valid_price], inplace=True)
    
    return price_df

if __name__ == "__main__":
    # Import the portfolio lists from assetsRoster (only these three are used)
    from geckoAPI.assetsRoster import carteira_EXC, carteira_HB, carteira_LC

    # (1) Pull allocation data from Google Sheets and parse the percentages and allocation date.
    latest_allocations = get_latest_allocations()
    
    exc_alloc, exc_date = latest_allocations.get('carteira_EXC', ({}, None))
    hb_alloc, hb_date = latest_allocations.get('carteira_HB', ({}, None))
    lc_alloc, lc_date = latest_allocations.get('carteira_LC', ({}, None))
    
    # (3) Build a price close DataFrame for each portfolio using only non-zero allocations.
    # Each portfolio's price data will start on the allocation date extracted from the sheet.
    exc_price_df = build_dataframe(carteira_EXC, exc_alloc)
    hb_price_df = build_dataframe(carteira_HB, hb_alloc)
    lc_price_df = build_dataframe(carteira_LC, lc_alloc)
    
    # Print (or further process) the resulting DataFrames
    print("EXC Portfolio Price Data:")
    print(exc_price_df.head())
    
    print("\nHB Portfolio Price Data:")
    print(hb_price_df.head())
    
    print("\nLC Portfolio Price Data:")
    print(lc_price_df.head())

Found date: 2025-02-12 00:00:00
Found date: 2025-02-12 00:00:00
Found date: 2025-02-12 00:00:00
EXC Portfolio Price Data:
               BTC      ETH      LINK       AAVE       UNI       SOL  \
date                                                                   
2013-04-27  135.30  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-04-28  141.96  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-04-29  135.30  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-04-30  117.00  2.83162  0.225377  56.163203  3.443832  0.957606   
2013-05-01  103.43  2.83162  0.225377  56.163203  3.443832  0.957606   

              PENDLE       AKT    RENDER      ONDO      AERO    MORPHO  \
date                                                                     
2013-04-27  1.801588  0.402013  0.051188  0.219553  0.061438  1.268028   
2013-04-28  1.801588  0.402013  0.051188  0.219553  0.061438  1.268028   
2013-04-29  1.801588  0.402013  0.051188  0.219553  0.061438  1.268028   
201

In [6]:
from geckoAPI.assetsRoster import carteira_EXC, carteira_HB, carteira_LC, ROSTER

roster_price_df = build_dataframe(ROSTER)

roster_price_df



ValueError: cannot reindex on an axis with duplicate labels

In [15]:
import pandas_ta as ta
import plotly.express as px

def analyze_portfolio_technicals(price_df):
    """
    Calculates current drawdown, max historical drawdown, and RSI for each asset
    and creates a scatter plot of RSI vs Drawdown.
    """
    # Initialize DataFrames to store results
    rsi_df = pd.DataFrame()
    drawdown_df = pd.DataFrame()
    max_drawdowns = {}
    
    # Calculate RSI and Drawdown for each asset
    for column in price_df.columns:
        # Calculate 14-day RSI
        rsi_df[column] = ta.rsi(price_df[column], length=14)
        
        # Calculate Drawdown
        rolling_max = price_df[column].expanding().max()
        drawdown = (price_df[column] - rolling_max) / rolling_max * 100
        drawdown_df[column] = drawdown
        
        # Calculate max historical drawdown
        max_drawdowns[column] = drawdown.min()  # Changed from mean() to min()
    
    # Get current values
    current_rsi = rsi_df.iloc[-1]
    current_drawdown = drawdown_df.iloc[-1]
    
    # Create DataFrame for plotting
    plot_df = pd.DataFrame({
        'Asset': current_rsi.index,
        'RSI': current_rsi.values,
        'Drawdown': current_drawdown.values,
        'Max_Drawdown': [max_drawdowns[asset] for asset in current_rsi.index]
    })
    
    # Rest of the function remains the same
    
    # Create scatter plot with Drawdown on Y-axis
    fig = px.scatter(
        plot_df,
        x='RSI',
        y='Drawdown',
        text='Asset',
        title='Portfolio Assets: Drawdown vs RSI',
        labels={
            'RSI': '14-day RSI',
            'Drawdown': 'Current Drawdown (%)'
        }
    )
    
    # Update layout
    fig.update_traces(
        textposition='top center',
        marker=dict(size=10)
    )
    fig.update_layout(
        plot_bgcolor='white',
        showlegend=False,
        hovermode='closest',
        # Add RSI reference lines
        xaxis=dict(
            range=[0, 100],
            gridcolor='lightgrey',
            dtick=10,  # Set tick spacing to 10
            tick0=0,   # Start ticks at 0
            tickmode='linear'  # Force linear tick mode
        ),
        yaxis=dict(
            gridcolor='lightgrey'
        )
    )
    
    # Add RSI reference lines (vertical)
    fig.add_vline(x=30, line_dash="dash", line_color="green", opacity=0.5)
    fig.add_vline(x=70, line_dash="dash", line_color="red", opacity=0.5)
    
    # Show plot
    fig.show()
    
    return plot_df

# Usage example:
# 

In [17]:
analyze_portfolio_technicals(exc_price_df)
analyze_portfolio_technicals(hb_price_df)
analyze_portfolio_technicals(lc_price_df)

,Asset,RSI,Drawdown,Max_Drawdown
0,BOTTO,39.131151,-87.273415,-99.186256
1,PENDLE,51.201929,-48.471935,-98.479312
2,NTX,38.898215,-88.085181,-95.464294
3,SHDW,41.911077,-87.466617,-98.264941
4,VISTA,35.263138,-78.446890,-87.968854
5,ANON,44.114932,-65.688086,-70.767143
6,YNE,47.208493,-61.069139,-77.127379
